In [33]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI

In [38]:
url = 'https://startuplist.africa/location/lagos'

#initializing the document loader
#this is how you load a url for retrieval
loader = WebBaseLoader(url)

#this contains the langchain docs for retrieval
raw_docs = loader.load()


In [39]:
# Split the loaded web page text into smaller chunks of ~500 characters each.
# This creates multiple Document objects, making the text manageable for embedding, retrieval, and LLM processing.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500
)
text = text_splitter.split_documents(raw_docs)

In [40]:
# Create an OpenAI embedding object to convert text chunks into high-dimensional vectors
# These vectors represent the semantic meaning of each chunk for similarity search.
vector_embeddings = OpenAIEmbeddings()

In [41]:
# Create a FAISS vector database from text chunks.
# The from_documents method converts each Document into a vector embedding using vector_embeddings
# and stores the embeddings in FAISS for fast semantic search.
vector_db = FAISS.from_documents(text, vector_embeddings)

In [44]:
# Convert the FAISS vector database into a retriever object.
# The retriever allows easy semantic search: it takes a query, finds the most relevant chunks
# from the vector store, and returns them as Document objects for LLM processing.
retriever = vector_db.as_retriever()

In [59]:
#initializing the llm model
llm = ChatOpenAI(
    model='gpt-3.5-turbo',
    temperature=0.7,
    max_completion_tokens=100
)

#user question/inpput to the llm
query = input("Prompt:")

#relevant parts of the website relating to the query/question asked
relevant_docs = retriever.invoke(query)

#iterating through relevant docs to remove the chunk of page content needed
relevant_page_content = "\n\n".join([doc.page_content for doc in relevant_docs])

#stores a list of questions and answers previously generated by the llm
chat_history = []

#iterates through each question and answe
text_in_chat_history = "\n".join([f"User: {q}\nAssistant: {a}" for q, a in chat_history])

In [60]:
# Create a prompt for the LLM combining:
# 1. The formatted chat history
# 2. The relevant text chunks from retrieved documents
# 3. The current user question
# This ensures the LLM answers based on both prior conversation and factual information.
prompt = f"""
Use the chat history and information to answer the question:
Chat History = {text_in_chat_history}
Information = {relevant_page_content}
Question = {query}
"""

# Send the prompt to the LLM to generate a response
response = llm.invoke(prompt)

# Add the current query and LLM answer to the chat history
# This ensures multi-turn context is preserved for future questions
chat_history.append((query, response.content))

print(response.content)

IHS Towers is a startup based in Lagos that operates in the telecommunications infrastructure industry. They provide services such as building and maintaining cell towers for telecom companies.
